### Rebalancing of PatternReframe ND augmented data

In [1]:
from pathlib import Path

PROJECT_ROOT  = Path("../..")
PROCESSED_DIR = PROJECT_ROOT / "processed"
BAHASA_ND_CSV    = PROCESSED_DIR / "bahasa_nondistorted.csv"

print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"BAHASA_CSV:    {BAHASA_ND_CSV}")

PROCESSED_DIR: ../../processed
BAHASA_CSV:    ../../processed/bahasa_nondistorted.csv


In [2]:
import pandas as pd

SEED = 42

# --- 1. Load existing augmented splits (we keep distorted side and test set unchanged) ---
train_aug_A = pd.read_csv(PROCESSED_DIR / "patternreframe_train_aug_v1_balanced.csv")
val_aug_A   = pd.read_csv(PROCESSED_DIR / "patternreframe_val_aug_v1_balanced.csv")
test_aug_A  = pd.read_csv(PROCESSED_DIR / "patternreframe_test_aug_v1_balanced.csv")

# Split each augmented file into distorted side and ND side
def split_aug(df):
    nd = df[df["pattern"] == "No Distortion"].copy()
    d  = df[df["pattern"] != "No Distortion"].copy()
    return d, nd

train_d, train_nd_A = split_aug(train_aug_A)
val_d,   val_nd_A   = split_aug(val_aug_A)
test_d,  test_nd_A  = split_aug(test_aug_A)

print(f"Variant A: train ND = {len(train_nd_A)}, val ND = {len(val_nd_A)}, test ND = {len(test_nd_A)}")
print(f"Distorted (unchanged): train {len(train_d)}, val {len(val_d)}, test {len(test_d)}")

# --- 2. Load the full Bahasa ND pool, apply v1 filter, drop test-set records ---
bah_nd = pd.read_csv(BAHASA_ND_CSV)

# v1 length filter: [40, 150] chars
v1 = bah_nd[(bah_nd["text_english_length"] >= 40) & (bah_nd["text_english_length"] <= 150)].copy()
v1 = v1.drop_duplicates(subset="text_english").reset_index(drop=True)
print(f"\nBahasa ND pool after v1 filter & dedup: {len(v1)} records")

# Remove any ND records already in the test set — that's the leakage we must avoid
test_texts = set(test_nd_A["thought"].astype(str).tolist())
v1_pool = v1[~v1["text_english"].isin(test_texts)].copy().reset_index(drop=True)
print(f"Available for train+val (after removing test): {len(v1_pool)}")

# --- 3. Split the pool into train and val using the same train:val ratio as variant A ---
train_val_ratio = len(train_nd_A) / (len(train_nd_A) + len(val_nd_A))
v1_pool = v1_pool.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_train_new = int(round(len(v1_pool) * train_val_ratio))
train_nd_B  = v1_pool.iloc[:n_train_new].copy()
val_nd_B    = v1_pool.iloc[n_train_new:].copy()

print(f"\nVariant B: train ND = {len(train_nd_B)}, val ND = {len(val_nd_B)}, test ND = {len(test_nd_A)} (unchanged)")

# --- 4. Convert Bahasa rows to the same schema as the PatternReframe distorted rows ---
# We only need 'thought' and 'pattern' for the binary task, so fill those.
# Other columns will be NaN; that's fine for the binary detector which only reads
# 'thought' and 'pattern'.
def to_pr_schema(bah_df):
    out = pd.DataFrame()
    out["thought"] = bah_df["text_english"].values
    out["pattern"] = "No Distortion"
    return out

train_nd_B = to_pr_schema(train_nd_B)
val_nd_B   = to_pr_schema(val_nd_B)

# --- 5. Assemble variant B splits: original distorted + new ND ---
# Concatenate, shuffle, save.
train_B = pd.concat([train_d, train_nd_B], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_B   = pd.concat([val_d,   val_nd_B],   ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
# Test stays exactly as variant A — that's the whole point
test_B = test_aug_A.copy()

# --- 6. Save with a clear suffix ---
train_B.to_csv(PROCESSED_DIR / "patternreframe_train_aug_v1_rebalanced.csv", index=False)
val_B.to_csv(PROCESSED_DIR / "patternreframe_val_aug_v1_rebalanced.csv", index=False)
test_B.to_csv(PROCESSED_DIR / "patternreframe_test_aug_v1_rebalanced.csv", index=False)

print(f"\n=== Summary ===")
print(f"Variant A (existing): {len(train_aug_A)}/{len(val_aug_A)}/{len(test_aug_A)}")
print(f"  Train: distorted {len(train_d)} | ND {len(train_nd_A)} ({len(train_nd_A)/(len(train_d)+len(train_nd_A))*100:.1f}%)")
print(f"  Val:   distorted {len(val_d)}   | ND {len(val_nd_A)} ({len(val_nd_A)/(len(val_d)+len(val_nd_A))*100:.1f}%)")
print(f"Variant B (rebalanced): {len(train_B)}/{len(val_B)}/{len(test_B)}")
print(f"  Train: distorted {len(train_d)} | ND {len(train_nd_B)} ({len(train_nd_B)/len(train_B)*100:.1f}%)")
print(f"  Val:   distorted {len(val_d)}   | ND {len(val_nd_B)} ({len(val_nd_B)/len(val_B)*100:.1f}%)")
print(f"Test split is identical for both variants ({len(test_B)} rows).")
print(f"\nSaved variant B splits to {PROCESSED_DIR}")

Variant A: train ND = 192, val ND = 96, test ND = 681
Distorted (unchanged): train 1920, val 961, test 6807

Bahasa ND pool after v1 filter & dedup: 1285 records
Available for train+val (after removing test): 604

Variant B: train ND = 403, val ND = 201, test ND = 681 (unchanged)

=== Summary ===
Variant A (existing): 2112/1057/7488
  Train: distorted 1920 | ND 192 (9.1%)
  Val:   distorted 961   | ND 96 (9.1%)
Variant B (rebalanced): 2323/1162/7488
  Train: distorted 1920 | ND 403 (17.3%)
  Val:   distorted 961   | ND 201 (17.3%)
Test split is identical for both variants (7488 rows).

Saved variant B splits to ../../processed
